[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/03_real_world_io/A2_duckdb_polars.ipynb)

> 📎 **Appendix notebook — reference style.** One of the optional appendices (see `README.md`): a demo/reference tour rather than a full lesson. The Parquet sections run on the **core course stack** (`pandas` + `pyarrow`). The **DuckDB** and **Polars** sections use the real libraries when installed (`pip install duckdb polars` — small, pure-Python-wheel installs, no server, no account) and **skip gracefully with a printed note when they aren't**. Everything is generated inline and runs 100 % offline.

---
# 📓 Notebook A2 (I/O) — When pandas Isn't Enough: Parquet, DuckDB & Polars

> **Module:** Real-world I/O · **Type:** Appendix · **Estimated time:** ~60–75 min · **Difficulty:** Intermediate

NB 7 taught you pandas; NB 13 taught you SQL against a tidy SQLite database. This appendix is about the day the data outgrows both habits: the log file is 6 million rows, every rerun starts with seconds of CSV parsing, the laptop fan spins up, and someone mutters "we need Spark." Almost always, you don't. Between "pandas on a CSV" and "a data warehouse" sits a modern middle layer — **a columnar file format (Parquet)** and **two single-machine engines (DuckDB and Polars)** — that handles tens of millions of rows on the machine you already have.

> 🧠 **Mental model.** Three separate dials got conflated in "pandas is slow": the **file format** (row-oriented CSV vs columnar Parquet), the **execution model** (eager single-core vs lazy multi-core with pushdown), and the **memory model** (everything in RAM vs streaming). You can upgrade each dial independently — and the *first* one is usually worth more than the other two combined.

## 🎯 Learning objectives

- Say precisely **why CSV is the bottleneck** (row layout, text parsing, no column/row skipping) and switch a workload to **Parquet** with two lines.
- Query Parquet files **directly with SQL** via DuckDB — including a whole directory of files at once — and hand results back to pandas.
- Read and write the **Polars lazy API**, and inspect a query plan to see **projection & predicate pushdown** actually happening.
- Run an honest micro-benchmark, and read it **without fooling yourself**.
- Choose between pandas, DuckDB, Polars, and a real warehouse using the two questions that matter: *does it fit in RAM?* and *what shape is the workload?*

## ✅ Prerequisites

**NB 7** (pandas fundamentals) and **NB 13** (SQL — DuckDB speaks the same language). `pyarrow` ships with the course requirements; DuckDB/Polars are optional installs.

## 📦 Install

```bash
pip install duckdb polars     # optional — this notebook skips those sections cleanly without them
```

In [1]:
import numpy as np
import pandas as pd
import os, pathlib, tempfile, time

try:
    import duckdb
    HAS_DUCKDB = True
except ImportError:
    HAS_DUCKDB = False
try:
    import polars as pl
    HAS_POLARS = True
except ImportError:
    HAS_POLARS = False

print(f"pandas {pd.__version__}  |  duckdb: {'yes' if HAS_DUCKDB else 'NOT INSTALLED (sections skip)'}"
      f"  |  polars: {'yes' if HAS_POLARS else 'NOT INSTALLED (sections skip)'}")

pandas 3.0.3  |  duckdb: yes  |  polars: yes


## 1. The morning the notebook stopped fitting

The webshop from NB 13 grew up. Its order-line log is now **6 million rows a year**, and the analytics team's ritual — `pd.read_csv(...)` then groupby — has started to hurt. Let's rebuild that situation honestly: generate the year, write it as the CSV the team currently uses, and feel the pain with a stopwatch running.

In [2]:
rng = np.random.default_rng(42)
N = 6_000_000

ts = pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365 * 24 * 3600, N), unit="s")
CATS = np.array(["electronics", "home", "sports", "beauty", "toys", "grocery"])
COUNTRIES = np.array(["DE", "AT", "CH", "NL", "FR"])

events = pd.DataFrame({
    "order_id": np.arange(N),
    "ts": ts,
    "customer_id": rng.integers(1, 120_000, N),
    "category": CATS[rng.integers(0, len(CATS), N)],
    "country": COUNTRIES[rng.choice(len(COUNTRIES), N, p=[0.55, 0.15, 0.10, 0.10, 0.10])],
    "qty": rng.integers(1, 5, N),
    "unit_price": (rng.gamma(2.0, 14.0, N) + 2).round(2),
})
print(f"{len(events):,} rows -- {events.memory_usage(deep=True).sum() / 1e6:,.0f} MB in RAM")
events.head(3)

6,000,000 rows -- 386 MB in RAM


,order_id,ts,customer_id,category,country,qty,unit_price
0,0,2025-02-02 13:50:18,94343,electronics,DE,1,11.91
1,1,2025-10-10 11:51:17,82691,grocery,NL,2,24.26
2,2,2025-08-27 22:02:47,14378,home,NL,2,24.03


In [3]:
DATA = pathlib.Path(tempfile.mkdtemp(prefix="a2_io_"))   # scratch dir; auto-cleaned by the OS

t0 = time.perf_counter()
events.to_csv(DATA / "events.csv", index=False)
t_csv_write = time.perf_counter() - t0

t0 = time.perf_counter()
csv_roundtrip = pd.read_csv(DATA / "events.csv", parse_dates=["ts"])
t_csv_read = time.perf_counter() - t0

size_csv = os.path.getsize(DATA / "events.csv") / 1e6
print(f"CSV: {size_csv:,.0f} MB on disk -- write {t_csv_write:.1f}s, read back {t_csv_read:.1f}s")
del csv_roundtrip

CSV: 312 MB on disk -- write 5.8s, read back 2.6s


A third of a gigabyte, and seconds of wall-clock in *each* direction — before a single line of analysis. Why is CSV so expensive? Because it stores **rows as text**:

```
   CSV (row-oriented text)                 Parquet (columnar binary)
   ┌───────────────────────────┐           ┌─────────┬─────────┬─────────┐
   │ 17,2025-03-01,DE,toys,...  │           │order_id │ country │  qty ...│
   │ 18,2025-03-01,AT,home,...  │           │ (ints,  │ (dict-  │ (ints,  │
   │ 19,2025-03-02,DE,toys,...  │           │ packed) │ encoded)│ packed) │
   │  ... 6M more lines ...     │           ├─────────┴─────────┴─────────┤
   └───────────────────────────┘           │ + per-chunk min/max statistics│
   read = parse EVERY byte,                └──────────────────────────────┘
   every row, every column                  read = ONLY the columns you ask,
                                            skip chunks the stats rule out
```

Every number is re-parsed from text on every read; you cannot read *one column* without scanning all of them; there's no compression and no statistics. Three structural problems, one fix.

## 2. Fix the format first: Parquet

**Parquet** is a columnar, compressed, binary format with per-chunk min/max statistics — and pandas already speaks it via `pyarrow`. Same DataFrame, same API, two changed lines:

In [4]:
t0 = time.perf_counter()
events.to_parquet(DATA / "events.parquet", index=False)
t_pq_write = time.perf_counter() - t0

t0 = time.perf_counter()
pq_roundtrip = pd.read_parquet(DATA / "events.parquet")
t_pq_read = time.perf_counter() - t0
del pq_roundtrip

size_pq = os.path.getsize(DATA / "events.parquet") / 1e6
print(f"{'':12}{'size on disk':>14}{'write':>9}{'read':>9}")
print(f"{'CSV':12}{size_csv:>11,.0f} MB{t_csv_write:>8.1f}s{t_csv_read:>8.1f}s")
print(f"{'Parquet':12}{size_pq:>11,.0f} MB{t_pq_write:>8.1f}s{t_pq_read:>8.1f}s")
print(f"\nParquet is ~{size_csv / size_pq:.0f}x smaller and ~{t_csv_read / t_pq_read:.0f}x faster to read -- same pandas code.")

              size on disk    write     read
CSV                 312 MB     5.8s     2.6s
Parquet             105 MB     0.7s     0.1s

Parquet is ~3x smaller and ~27x faster to read -- same pandas code.


In [5]:
# The columnar superpower: read ONLY what the question needs.
t0 = time.perf_counter()
two_cols = pd.read_parquet(DATA / "events.parquet", columns=["category", "unit_price"])
t_two = time.perf_counter() - t0
print(f"all 7 columns : {t_pq_read:.2f}s")
print(f"just 2 columns: {t_two:.2f}s   <- impossible with CSV: rows force you to parse everything")
del two_cols

all 7 columns : 0.10s
just 2 columns: 0.04s   <- impossible with CSV: rows force you to parse everything


> 💡 **The cheapest big-data upgrade in this course:** before installing anything new, convert the file. Same pandas, same code — here ~3× on disk and >20× on reads. If your pipeline still fits in RAM after that, **you're done** — the engines below are for when it doesn't, or when the pipeline itself gets heavy.

---

### ✋ Quick exercise (~2 min) — why can't CSV do that?

The `columns=["category", "unit_price"]` trick cut the read time by a large factor. Explain *mechanically* why the same argument can't exist for `pd.read_csv` — what would the parser still have to do? And which Parquet feature would let a reader skip most of the file for the query `unit_price > 500`?

In [6]:
# ✍️ Your turn 👇  (reasoning exercise -- answer in a comment)
# 1. why no columns= shortcut for CSV:
# 2. which Parquet feature helps `unit_price > 500`:

<details>
<summary>✅ <b>Solution</b></summary>

```python
print("1. A CSV row is one text line; column boundaries only exist after parsing the line.")
print("   To extract 2 columns the parser still reads and splits ALL bytes of ALL rows -- ")
print("   the work you wanted to skip is the work that finds the columns.")
print("2. Per-chunk MIN/MAX statistics: if a row-group's max(unit_price) is 480, the whole")
print("   chunk is skipped without decompressing it -- 'predicate pushdown'.")
```

That's the whole architecture lesson: CSV couples *reading* to *parsing everything*; Parquet's layout plus statistics let engines read only the columns you project and the chunks that can possibly match your filter. Everything DuckDB and Polars do fast below is built on exactly those two skips.
</details>

## 3. DuckDB — the SQL you already know, pointed at files

NB 13 gave you SQL against a SQLite database *you had to load first*. **DuckDB** is the same idea with two twists: it's built for analytics (columnar, vectorized, multi-core), and it treats **files as tables** — you `SELECT ... FROM 'events.parquet'` directly, no import step, no server, no account. It's a `pip install`, and it has become the default way data scientists run SQL on local files.

In [7]:
if HAS_DUCKDB:
    revenue_q4 = duckdb.sql(f"""
        SELECT category,
               ROUND(SUM(qty * unit_price)) AS revenue
        FROM '{DATA / "events.parquet"}'
        WHERE country = 'DE' AND datepart('quarter', ts) = 4
        GROUP BY category
        ORDER BY revenue DESC
    """).df()                     # .df() hands the result straight back to pandas
    print(revenue_q4)
else:
    print("duckdb not installed -- skipping (pip install duckdb)")

      category     revenue
0  electronics  10422126.0
1         toys  10401121.0
2       sports  10394809.0
3       beauty  10386420.0
4         home  10365415.0
5      grocery  10330243.0


In [8]:
# It scales past one file: a glob is a table too. Write monthly partition files
# (how event data usually lands from an export job), then query December + November
# + October by PATTERN -- files that don't match are never opened.
part_dir = DATA / "events_monthly"; part_dir.mkdir(exist_ok=True)
for m, g in events.groupby(events.ts.dt.month):
    g.to_parquet(part_dir / f"month={m:02d}.parquet", index=False)

if HAS_DUCKDB:
    q4 = duckdb.sql(f"""
        SELECT COUNT(*) AS orders, ROUND(SUM(qty * unit_price)) AS revenue
        FROM '{part_dir}/month=1*.parquet'
    """).df()
    print("Q4, straight from three of twelve files:")
    print(q4)
else:
    print("duckdb not installed -- skipping")

Q4, straight from three of twelve files:
    orders      revenue
0  1512615  113485484.0


Notice what did **not** happen: no `CREATE TABLE`, no load step, and — for the glob query — nine of the twelve files were never touched. DuckDB also *streams*: it can aggregate files larger than your RAM, because it never needs the whole table in memory at once. That is the line pandas cannot cross, no matter the format.

> 🔗 **Interop is free.** `.df()` converts results to pandas (via Arrow, mostly zero-copy), so DuckDB slots *into* your existing workflow: heavy lifting in SQL, then seaborn/sklearn on the small result exactly as in NB 7–13. You can even query an existing DataFrame: `duckdb.sql("SELECT ... FROM events")` finds the variable by name.

---

### ✋ Quick exercise (~2 min) — average order value by country

Write the DuckDB query: **average order value** (`qty * unit_price`, averaged per order line) by `country`, sorted descending, straight from `events.parquet`. (If DuckDB isn't installed, write the SQL in the comment anyway — it's NB 13 SQL.)

In [9]:
# ✍️ Your turn 👇
# SELECT country, ... FROM '<the parquet file>' GROUP BY ... ORDER BY ...

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DUCKDB:
    aov = duckdb.sql(f'''
        SELECT country,
               ROUND(AVG(qty * unit_price), 2) AS avg_order_value
        FROM '{DATA / "events.parquet"}'
        GROUP BY country
        ORDER BY avg_order_value DESC
    ''').df()
    print(aov)
else:
    print("SQL: SELECT country, ROUND(AVG(qty * unit_price), 2) AS aov "
          "FROM 'events.parquet' GROUP BY country ORDER BY aov DESC")
```

Nothing new beyond NB 13 except the `FROM 'file.parquet'` — which is the point. The values land almost on top of each other across countries (the generator draws prices independently of country), which is itself a useful sanity check: when a groupby shows structure you didn't plant, ask the NB 10 question — is it real or is it noise?
</details>

## 4. Polars — the pipeline that plans before it runs

**Polars** is a DataFrame library like pandas, but built lazy-first and multi-core. In the lazy API you don't execute operations — you *describe* them, and Polars builds a **query plan** it optimizes before touching the file: only needed columns are read (projection pushdown), filters run at the scan (predicate pushdown), and the whole plan runs across cores. The grammar: `scan_*` → chain expressions → `.collect()`.

In [10]:
if HAS_POLARS:
    lazy_q4 = (
        pl.scan_parquet(DATA / "events.parquet")                # scan, don't read
          .filter((pl.col("country") == "DE") & (pl.col("ts").dt.quarter() == 4))
          .group_by("category")
          .agg((pl.col("qty") * pl.col("unit_price")).sum().round(0).alias("revenue"))
          .sort("revenue", descending=True)
    )
    print(type(lazy_q4).__name__, "-- nothing has executed yet. The plan:")
    print(lazy_q4.explain())
else:
    print("polars not installed -- skipping (pip install polars)")

LazyFrame -- nothing has executed yet. The plan:
SORT BY [descending: [true]] [col("revenue")]
  AGGREGATE[maintain_order: false]
    [(col("qty").cast(Float64) * col("unit_price")).sum().round().alias("revenue")] BY [col("category")]
    FROM
    simple π 3/3 ["category", "qty", ... 1 other column]
      Parquet SCAN [/var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/a2_io_hxy7wb8p/events.parquet]
      PROJECT 5/7 COLUMNS
      SELECTION: (col("country") == "DE") & (col("ts").dt.quarter() == 4)
      ESTIMATED ROWS: 6000000


In [11]:
if HAS_POLARS:
    print(lazy_q4.collect())      # NOW it runs -- optimized, multi-core
else:
    print("polars not installed -- skipping")

shape: (6, 2)
┌─────────────┬─────────────┐
│ category    ┆ revenue     │
│ ---         ┆ ---         │
│ str         ┆ f64         │
╞═════════════╪═════════════╡
│ electronics ┆ 1.0422126e7 │
│ toys        ┆ 1.0401121e7 │
│ sports      ┆ 1.0394809e7 │
│ beauty      ┆ 1.038642e7  │
│ home        ┆ 1.0365415e7 │
│ grocery     ┆ 1.0330243e7 │
└─────────────┴─────────────┘


Read the plan bottom-up and you can *see* the two pushdowns from §2's solution: `PROJECT` lists a handful of the 7 columns (only what the query touches leaves the file), and `SELECTION` shows the country filter running inside the Parquet scan. You reasoned about that architecture in the checkpoint; `explain()` is where it stops being theory.

The other adjustment coming from pandas is the **expression** style — you describe columns (`pl.col("qty") * pl.col("unit_price")`) instead of operating on materialized Series. A translation table for the moves you already know:

| pandas (NB 7) | Polars (lazy) |
|---|---|
| `pd.read_parquet(f)` | `pl.scan_parquet(f)` … `.collect()` |
| `df[df.country == "DE"]` | `.filter(pl.col("country") == "DE")` |
| `df.assign(rev=df.qty * df.unit_price)` | `.with_columns((pl.col("qty") * pl.col("unit_price")).alias("rev"))` |
| `df.groupby("cat")["rev"].sum()` | `.group_by("cat").agg(pl.col("rev").sum())` |
| `df.sort_values("rev", ascending=False)` | `.sort("rev", descending=True)` |
| `df.head(3)` per group: `groupby(...).head(3)` | `.group_by(...).head(3)` |
| back to pandas | `.collect().to_pandas()` |

---

### ✋ Quick exercise (~2 min) — translate one chain

Translate this NB 7-style pandas into a single lazy Polars chain (don't forget the last step!):

```python
out = (events[events.qty >= 3]
       .groupby("country")["unit_price"].mean()
       .round(2)
       .sort_values(ascending=False))
```

In [12]:
# ✍️ Your turn 👇
# out_pl = (pl.scan_parquet(DATA / "events.parquet") ... )

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_POLARS:
    out_pl = (
        pl.scan_parquet(DATA / "events.parquet")
          .filter(pl.col("qty") >= 3)
          .group_by("country")
          .agg(pl.col("unit_price").mean().round(2).alias("unit_price"))
          .sort("unit_price", descending=True)
          .collect()
    )
    print(out_pl)
else:
    print("polars not installed -- the chain: scan_parquet -> filter(qty>=3) -> "
          "group_by(country).agg(mean unit_price) -> sort -> collect()")
```

The step everyone forgets is **`.collect()`** — without it you have a plan, not a result (Exercise 3 below turns that mistake into a benchmark scandal). Note also that the aggregation needs an explicit `.alias(...)`; Polars won't silently reuse the input name for a derived column the way pandas broadcasting sometimes lets you get away with.
</details>

## 5. The benchmark — and how to read one honestly

Same question four ways — *Q4 revenue by category for Germany* — timed cold from the file each time. Then the honest reading, which matters more than the numbers.

In [13]:
def bench(label, fn, results={}):
    t0 = time.perf_counter(); out = fn(); dt = time.perf_counter() - t0
    print(f"{label:22s} {dt:6.2f} s")
    return out

print(f"{'engine':22s} {'time':>8}")
r_csv = bench("pandas <- CSV", lambda: (lambda d: (d.qty * d.unit_price)
              .groupby(d.category).sum())(pd.read_csv(DATA / "events.csv", parse_dates=["ts"])
              .query("country == 'DE' and ts.dt.quarter == 4")))
r_pd = bench("pandas <- Parquet", lambda: (lambda d: (d.qty * d.unit_price)
             .groupby(d.category).sum())(pd.read_parquet(DATA / "events.parquet")
             .query("country == 'DE' and ts.dt.quarter == 4")))
if HAS_DUCKDB:
    r_dk = bench("DuckDB <- Parquet", lambda: duckdb.sql(f"""
        SELECT category, SUM(qty * unit_price) AS revenue
        FROM '{DATA / "events.parquet"}'
        WHERE country = 'DE' AND datepart('quarter', ts) = 4
        GROUP BY category""").df())
if HAS_POLARS:
    r_pl = bench("Polars lazy <- Parquet", lambda: (
        pl.scan_parquet(DATA / "events.parquet")
          .filter((pl.col("country") == "DE") & (pl.col("ts").dt.quarter() == 4))
          .group_by("category")
          .agg((pl.col("qty") * pl.col("unit_price")).sum().alias("revenue"))
          .collect()))

engine                     time


pandas <- CSV            2.58 s


pandas <- Parquet        0.27 s
DuckDB <- Parquet        0.12 s
Polars lazy <- Parquet   0.03 s


In [14]:
# Never trust a benchmark whose engines might disagree on the ANSWER.
check = r_pd.round(0).sort_index()
if HAS_DUCKDB:
    dk = r_dk.set_index("category")["revenue"].round(0).sort_index()
    print("pandas == duckdb :", bool(np.allclose(check, dk)))
if HAS_POLARS:
    plr = r_pl.to_pandas().set_index("category")["revenue"].round(0).sort_index()
    print("pandas == polars :", bool(np.allclose(check, plr)))

pandas == duckdb : True
pandas == polars : True


On this machine the ladder is roughly: **CSV → Parquet buys the big jump; Parquet → an engine buys another ~2–9×.** Your machine will print different numbers — and that's the first of four honest-reading rules:

1. **One machine, one run, warm OS cache** — this is a *micro*-benchmark. Directionally robust, decimal-place meaningless.
2. **The format did the heavy lifting.** The pandas row already fell from seconds to sub-second by switching files. If someone benchmarks `polars` against `pandas.read_csv`, they're selling you the format change as an engine change.
3. **At 6M rows everything fits in RAM,** so the engines' deepest advantage — streaming larger-than-memory data — isn't even being measured here. The gap *widens* with scale.
4. **Engines shine on complex plans.** A single filter+groupby is pandas' best case; joins, windows, and multi-step pipelines are where the optimizer and the extra cores really pay (compare Exercise 2).

## 6. Which tool, when

Two questions decide almost every case: **does it fit in RAM (with headroom for copies)?** and **what shape is the work?**

| Situation | Reach for | Why |
|---|---|---|
| < ~1M rows, exploratory, plots & sklearn | **pandas** | The ecosystem is the feature; nothing else integrates as widely |
| Same, but files are big or slow to load | **pandas + Parquet** | 2-line change, ~10× I/O, zero new APIs |
| SQL-shaped questions over files (possibly many, possibly > RAM) | **DuckDB** | NB 13 skills reused verbatim; files-as-tables; streams past RAM |
| Pipeline-shaped transforms, performance matters, > a few M rows | **Polars (lazy)** | Optimizer + all cores + pushdown; pandas-like feel |
| Many users / dashboards / permissions / always-on | a real **warehouse** (BigQuery, Snowflake, Postgres) | Concurrency, governance and uptime are organizational problems, not file-format problems |

Two closing notes on the ecosystem: everything above meets in **Apache Arrow** (the shared in-memory format — it's why `.df()`, `.to_pandas()` and friends are cheap), and the boundaries blur — pandas can use Arrow dtypes, DuckDB can query Polars frames, Polars can run SQL. You're not choosing a religion; you're choosing the ergonomics for this job.

> ⚠️ **The trap to avoid** is the one this appendix opened with: reaching for a cluster because a CSV was slow. The 2020s version of "we need Spark" is usually solved by one file-format change and one `pip install` — and your laptop, which has no cluster to queue for, will finish first.

## 🧪 Exercises

Reference-style appendix, so these are a short numbered list (no ⭐ ratings or 🎁 mini-project). Each has a worked solution.

**Exercise 1 — Measure the column pruning.** Time `pd.read_parquet` on `events.parquet` reading (a) all columns, (b) `["qty", "unit_price"]`, (c) just `["country"]`. Relate the three times to the on-disk layout from §1's diagram.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
for cols in (None, ["qty", "unit_price"], ["country"]):
    t0 = time.perf_counter()
    _ = pd.read_parquet(DATA / "events.parquet", columns=cols)
    label = "all 7" if cols is None else str(cols)
    print(f"{label:28s} {time.perf_counter() - t0:5.2f} s")
```

Times fall roughly with the bytes actually read: each Parquet column lives in its own contiguous, compressed chunks, so unread columns are unread *bytes*. One instructive wrinkle: the two *numeric* columns typically beat lone `country` even though `country` is tiny on disk (five distinct values dictionary-encode into almost nothing) — because after reading, `country` must be materialized as millions of Python-level strings, and decoding costs more than reading two tightly-packed numeric columns. On-disk size and in-memory cost are two different bills.
</details>

**Exercise 2 — A query that earns the engine.** Top-3 customers by revenue *per country* — a groupby (720k groups) plus a per-group ranking. Do it in pandas from Parquet, then in DuckDB with a window function (`ROW_NUMBER() OVER (PARTITION BY country ORDER BY revenue DESC)` — NB 13's window tour), time both, and check the answers agree.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
t0 = time.perf_counter()
d = pd.read_parquet(DATA / "events.parquet", columns=["customer_id", "country", "qty", "unit_price"])
rev = (d.assign(revenue=d.qty * d.unit_price)
         .groupby(["country", "customer_id"], as_index=False)["revenue"].sum())
top_pd = (rev.sort_values("revenue", ascending=False).groupby("country").head(3)
             .sort_values(["country", "revenue"], ascending=[True, False]).reset_index(drop=True))
print(f"pandas: {time.perf_counter() - t0:.2f} s")

if HAS_DUCKDB:
    t0 = time.perf_counter()
    top_dk = duckdb.sql(f'''
        WITH rev AS (
            SELECT country, customer_id, SUM(qty * unit_price) AS revenue
            FROM '{DATA / "events.parquet"}'
            GROUP BY country, customer_id)
        SELECT country, customer_id, revenue FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY country ORDER BY revenue DESC) AS rk
            FROM rev)
        WHERE rk <= 3
        ORDER BY country, revenue DESC''').df()
    print(f"duckdb: {time.perf_counter() - t0:.2f} s")
    print("same answer:", bool(np.allclose(top_pd.revenue.round(2), top_dk.revenue.round(2))))
else:
    print("duckdb not installed -- pandas-only for this one")
```

Expect DuckDB to keep (or widen) its lead here *even though* the pandas version got the same column-pruning trick — multi-step plans (aggregate → window → filter) are where a query optimizer and vectorized execution pull away, and the gap grows with data size and plan depth. Also worth noticing: the pandas version needed a careful `sort_values(...).groupby(...).head(3)` idiom, while the SQL states the intent directly. Fluency in *both* is the real skill.
</details>

**Exercise 3 — Debug me 🐞: the 3000× speedup.** A colleague benchmarks Polars against pandas and posts astonishing news:

```python
t0 = time.perf_counter()
result_pl = (pl.scan_parquet(DATA / "events.parquet")
               .filter(pl.col("country") == "DE")
               .group_by("category").agg(pl.col("qty").sum()))
print(f"polars: {time.perf_counter() - t0:.4f}s")     # 0.0002s !!
```

*"Polars is thousands of times faster than pandas!"* Run it, find the bug, fix the benchmark, and state the honest speedup.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
if HAS_POLARS:
    t0 = time.perf_counter()
    result_pl = (pl.scan_parquet(DATA / "events.parquet")
                   .filter(pl.col("country") == "DE")
                   .group_by("category").agg(pl.col("qty").sum()))
    t_plan = time.perf_counter() - t0
    print(f"'benchmark': {t_plan:.4f}s -- but look: {type(result_pl).__name__}")

    t0 = time.perf_counter()
    _ = result_pl.collect()
    print(f"with .collect(): {time.perf_counter() - t0:.2f}s -- the real number")
else:
    print("polars not installed -- the bug: no .collect(), so the 'result' is a LazyFrame;")
    print("the colleague timed BUILDING THE PLAN, not running it.")
```

The colleague timed **building the query plan** — the lazy chain executes nothing until `.collect()`. The type gives it away: `LazyFrame`, not `DataFrame`. The honest speedup over pandas-from-Parquet is meaningful but boring (§5's few-×), which is exactly why the wrong version spread further. The general habit, which will save you in every future benchmark: **time the code that produces the *answer*, and look at the answer** — lazy APIs, async calls, generators and cached properties all produce this same illusion.
</details>

**Exercise 4 — The migration function.** Write `csv_to_parquet(csv_path, parquet_path)` that converts a file, verifies the round trip (`shape` and dtypes-compatible equality on a numeric checksum), and returns the compression ratio. Run it on `events.csv`.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def csv_to_parquet(csv_path, parquet_path, parse_dates=("ts",)):
    d = pd.read_csv(csv_path, parse_dates=list(parse_dates))
    d.to_parquet(parquet_path, index=False)
    back = pd.read_parquet(parquet_path)
    assert back.shape == d.shape, "shape changed!"
    a = d.select_dtypes("number").sum().sum()
    b = back.select_dtypes("number").sum().sum()
    assert np.isclose(a, b), "numeric checksum changed!"
    ratio = os.path.getsize(csv_path) / os.path.getsize(parquet_path)
    return ratio

ratio = csv_to_parquet(DATA / "events.csv", DATA / "events_migrated.parquet")
print(f"verified round trip -- Parquet is {ratio:.1f}x smaller")
```

The checks matter more than the conversion: a real migration script that doesn't verify the round trip is how a silent `dtype` surprise (a numeric column with one stray string, a timezone shift) gets baked into a year of downstream jobs. For production, add a row-count log line and keep the CSV until the first weekly job succeeds off the new file.
</details>

## 🧠 Key takeaways

- **CSV is the bottleneck more often than pandas is.** Columnar + compressed + statistics (Parquet) buys the biggest single jump with a two-line change — make that move before adopting any new engine.
- **DuckDB = NB 13's SQL, pointed at files.** No server, no load step, globs over partitioned files, streams past RAM, `.df()` back to pandas.
- **Polars = the lazy pipeline.** Describe, let it optimize (watch `explain()` show projection & predicate pushdown), then `.collect()` — and *only* what's collected has actually run.
- **Read benchmarks honestly:** verify answers agree, separate format wins from engine wins, and remember that in-RAM microbenchmarks understate the engines' real advantage (streaming, complex plans).
- **Escalation ladder:** pandas → +Parquet → DuckDB/Polars → warehouse. Each step is justified by RAM or by workload shape — never by fashion.

## 🚀 Next step

You now have the *storage and speed* half of real-world I/O. NB 14 (Module 4) picks up the other half — getting data that only exists on web pages. And when a pipeline built here needs to run every night rather than in a notebook, that's Module 13's job.